# Annotation test run (validation set)

Runs a small LLM annotation test against the human validation gold: five
validation dialogues, all prompt templates, one model. Logic lives in
`extension/scripts/` (`prompt_loader`, `extraction`, `scoring`). Every call
caches per dialogue to `extension/artifacts/extraction_cache/{split}/{model}/{prompt}/{id}.json`
and re-runs skip valid entries. No output-token cap is set; models use their
provider defaults, and realised cost is measured from OpenRouter usage
accounting per call.

**Before running:** `export OPENROUTER_API_KEY=...` in the launching terminal.


In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "extension" / "artifacts").exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
print("cwd:", os.getcwd(), "| key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | key set: True


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring

gold = load_dataset("extension/artifacts/annotation_dev_and_val_sets/validation_set.csv")
# the validation gold was carved from the MathDial train split and keeps its
# train indices, so its cache lives under the train split: when the full
# train set is annotated later, these 78 dialogues are already done and skip
DIALOGUES = extraction.dialogues_from(gold, split="train")
UNITS = scoring.units_by_dialogue(gold)
print(f"{len(DIALOGUES)} dialogues, {len(gold)} units")


78 dialogues, 544 units


In [3]:
TEST_MODEL = 'qwen/qwen3.7-max'         # any OpenRouter model slug
TEST_PROMPTS = ['P1_full_codebook']       # any subset of prompt stems, e.g.
                                          # ['P1_full_codebook','P2_condensed_codebook',
                                          #  'P3_minimal','P4_condensed_staged']
N_TEST_DIALOGUES = 10
MAX_WORKERS = 5            # parallel in-flight requests; 4-6 is polite to a pinned provider
TEST_PROVIDER = None   # fastest OpenRouter host for Kimi K3; None for free
                          # routing (set None when testing other models unless
                          # you have checked they are served by the named host)


In [5]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}; available: {prompt_loader.list_prompts()}"
print(f"test model: {TEST_MODEL} on dialogues "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x prompts {TEST_PROMPTS}\n")

import json as _json
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    extraction.generate_annotations(prompt, TEST_MODEL, TEST_DIALOGUES,
                                    provider=TEST_PROVIDER, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(TEST_MODEL, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"cost ${rec['cost_usd']:.4f}  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, TEST_MODEL, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split='train')
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| alpha {s['alpha']:.3f} | ${s['usd_per_dialogue']:.4f}/dialogue")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ) + "\n")

import pandas as pd
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'alpha',
                *family_f1_cols, 'usd_per_dialogue', 'latency_s']
summary = pd.DataFrame(test_rows)[summary_cols].round(3)
print(summary.to_string(index=False))


test model: qwen/qwen3.7-max on dialogues [1, 21, 35, 79, 143, 178, 255, 270, 275, 289] x prompts ['P1_full_codebook']

  P1_full_codebook (10 dialogues, 5 workers):
  21: cached
  35: cached
  1: cached
  143: cached
  178: cached
  255: cached
  270: cached
  275: cached
  289: cached
  79: ok
  P1_full_codebook       1: ok       cost $0.0791  latency 122.2s
  P1_full_codebook       21: ok       cost $0.0784  latency 109.6s
  P1_full_codebook       35: ok       cost $0.0703  latency 58.8s
  P1_full_codebook       79: ok       cost $0.0781  latency 106.1s
  P1_full_codebook       143: ok       cost $0.0750  latency 85.6s
  P1_full_codebook       178: ok       cost $0.0428  latency 139.1s
  P1_full_codebook       255: ok       cost $0.0377  latency 112.7s
  P1_full_codebook       270: ok       cost $0.0686  latency 146.9s
  P1_full_codebook       275: ok       cost $0.0442  latency 165.8s
  P1_full_codebook       289: ok       cost $0.0360  latency 114.0s
  -> validity 100% | macro-F1(

### Notes

Prompt files live in `extension/artifacts/annotation_prompts/`; dropping a
new `P*.md` there adds it to the run automatically. Every attempt is cached
with its raw output, full reasoning trace, validation errors, and usage,
so misreadings of the codebook can be diagnosed from the trace, and are
retried on the next execution. Scores on five dialogues are a plumbing check,
not a ranking.
